# Conceptos Fundamentales — LLMs y Generative AI

**Módulo 3 — Introducción a AI Engineering** | DSRP Machine Learning Engineering

---

> 📖 Este notebook es un **glosario de referencia** para los conceptos fundamentales de LLMs, Transformers y GenAI. Cada concepto incluye definición, matemática, visualización y código.

### Índice

**Fundamentos de Transformers:**
1. [Tokens y Tokenización](#1)
2. [Embeddings — representación vectorial](#2)
3. [Similitud coseno y distancia euclidiana](#3)
4. [Softmax — de logits a probabilidades](#4)
5. [Positional Encodings — codificar el orden](#5)
6. [Attention Mechanism — Query, Key, Value](#6)
7. [Multi-Head Attention](#7)
8. [Feed-Forward Networks (FFN)](#8)
9. [Layer Normalization y conexiones residuales](#9)

**Generación y Evaluación:**
10. [Autoregresión — generación token a token](#10)
11. [Sampling strategies — temperature, top-k, top-p](#11)
12. [Perplexity — métrica de evaluación](#12)
13. [Context Window y KV Cache](#13)

**Fine-tuning y Scaling:**
14. [Fine-tuning vs Prompting](#14)
15. [Scaling Laws — más grande = mejor](#15)

**APIs y Producción:**
16. [APIs REST y HTTP — consumir modelos como servicio](#16)
17. [Métricas de similitud vectorial — comparando embeddings](#17)
18. [Vector Databases — búsqueda eficiente de embeddings](#18)
19. [Caching — evitar trabajo redundante](#19)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine, euclidean
from scipy.special import softmax

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
np.set_printoptions(precision=3, suppress=True)


---
<a id='1'></a>
## 1. Tokens y Tokenización

**Definición:** La **tokenización** es el proceso de dividir texto en unidades pequeñas llamadas **tokens**. Un token puede ser una palabra completa, una subpalabra, un carácter o incluso un byte.

### ¿Por qué no usar palabras completas?

| Problema | Solución con subpalabras |
|---|---|
| Vocabulario gigante (millones de palabras) | Vocabulario manejable (30k-200k tokens) |
| Palabras fuera del vocabulario (OOV) | Cualquier palabra se puede representar |
| No maneja morfología (correr, corriendo, corrió) | Comparte raíz: `[corr][iendo]`, `[corr][ió]` |

### Algoritmos principales

**1. Byte-Pair Encoding (BPE)** — usado por GPT, LLaMA
- Empieza con caracteres individuales
- Iterativamente fusiona los pares más frecuentes
- Ejemplo: `"running"` → `["run", "ning"]` o `["runn", "ing"]` según frecuencia

**2. WordPiece** — usado por BERT
- Similar a BPE pero maximiza la verosimilitud del corpus
- Usa `##` para marcar continuaciones: `["run", "##ning"]`

**3. SentencePiece** — usado por T5, LLaMA
- Trata el texto como secuencia de bytes (no requiere pre-tokenización)
- Soporta cualquier idioma sin espacios (chino, japonés)

### Propiedades importantes

- **Vocabulario fijo:** típicamente 30k-200k tokens
- **Reversible:** tokens → texto original (casi siempre)
- **Dependiente del modelo:** cada modelo tiene su propio tokenizer
- **Impacto en costo:** más tokens = más caro (APIs cobran por token)

In [ ]:
# Simulación de tokenización BPE simplificada
texto = "running runner run"
print(f"Texto original: '{texto}'\\n")

# Paso 1: caracteres individuales
tokens_iniciales = list(texto.replace(" ", "_"))
print(f"Paso 1 — caracteres: {tokens_iniciales}\\n")

# Paso 2: fusionar pares frecuentes (simulado manualmente)
# En BPE real, se cuenta frecuencia y se fusiona iterativamente
vocab_bpe = {
    "r": "r", "u": "u", "n": "n", "i": "i", "g": "g", "e": "e",
    "run": "run", "ning": "ning", "ner": "ner", "_": "_",
}

# Tokenización final (simulada)
tokens_finales = ["run", "ning", "_", "run", "ner", "_", "run"]
print(f"Paso 2 — después de fusiones BPE: {tokens_finales}\\n")

# Cada token tiene un ID en el vocabulario
vocab_ids = {tok: i for i, tok in enumerate(sorted(vocab_bpe.keys()))}
ids = [vocab_ids[t] for t in tokens_finales]
print(f"IDs en el vocabulario: {ids}")
print(f"\\nVocabulario completo ({len(vocab_ids)} tokens): {vocab_ids}")

# Visualización: comparación de tamaños de vocabulario
fig, ax = plt.subplots(figsize=(8, 4))
metodos = ['Palabras\\ncompletas', 'Caracteres', 'BPE/WordPiece\\n(subpalabras)']
tamaños = [500000, 256, 50000]
colores = ['tomato', 'steelblue', 'green']
bars = ax.barh(metodos, tamaños, color=colores, alpha=0.8)
for bar, tam in zip(bars, tamaños):
    ax.text(tam + 10000, bar.get_y() + bar.get_height()/2,
            f'{tam:,}', va='center', fontweight='bold')
ax.set_xlabel('Tamaño del vocabulario')
ax.set_title('Comparación de métodos de tokenización')
ax.set_xscale('log')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()


---
<a id='16'></a>
## 16. APIs REST y HTTP — consumir modelos como servicio

**Definición:** Una **API (Application Programming Interface)** es una interfaz que permite que dos aplicaciones se comuniquen. **REST (Representational State Transfer)** es un estilo arquitectónico para diseñar APIs sobre HTTP.

### Componentes de una petición HTTP

```http
POST /v1/chat/completions HTTP/1.1
Host: api.openai.com
Authorization: Bearer sk-...
Content-Type: application/json

{"model": "gpt-4o-mini", "messages": [...]}
```

| Componente | Qué es | Ejemplo |
|---|---|---|
| **Método** | Acción a realizar | `GET`, `POST`, `PUT`, `DELETE` |
| **Path** | Recurso específico | `/v1/chat/completions` |
| **Headers** | Metadata | `Authorization`, `Content-Type` |
| **Body** | Datos enviados | JSON con parámetros |

### Métodos HTTP principales

| Método | Uso | Idempotente | Ejemplo |
|---|---|---|---|
| **GET** | Leer datos | ✅ | Obtener lista de modelos |
| **POST** | Crear/ejecutar | ❌ | Generar texto |
| **PUT** | Actualizar completo | ✅ | Actualizar configuración |
| **DELETE** | Eliminar | ✅ | Borrar un fine-tune |

### Códigos de estado HTTP

| Rango | Significado | Ejemplos |
|---|---|---|
| **2xx** | Éxito | 200 OK, 201 Created |
| **4xx** | Error del cliente | 400 Bad Request, 401 Unauthorized, 429 Too Many Requests |
| **5xx** | Error del servidor | 500 Internal Server Error, 503 Service Unavailable |

### JSON — formato de intercambio de datos

**JSON (JavaScript Object Notation)** es el formato estándar para enviar y recibir datos en APIs REST.

```json
{
  "model": "gpt-4o-mini",
  "messages": [
    {"role": "user", "content": "Hola"}
  ],
  "temperature": 0.7,
  "max_tokens": 100
}
```

**Tipos de datos en JSON:**
- `string`: `"texto"`
- `number`: `42`, `3.14`
- `boolean`: `true`, `false`
- `null`: `null`
- `array`: `[1, 2, 3]`
- `object`: `{"key": "value"}`

### Rate Limiting — control de tasa

Los proveedores limitan cuántas peticiones puedes hacer por minuto/hora para evitar abuso:

```
x-ratelimit-limit-requests: 10000       ← máximo por minuto
x-ratelimit-remaining-requests: 9999    ← cuántas te quedan
x-ratelimit-reset-requests: 1704067260  ← cuándo se resetea (Unix timestamp)
```

**Estrategia cuando llegas al límite:**
1. Detectar el error `429 Too Many Requests`
2. Leer el header `Retry-After` o `x-ratelimit-reset-requests`
3. Esperar ese tiempo (+ un poco de jitter aleatorio)
4. Reintentar

**Backoff exponencial:**

```python
import time
import random

def retry_with_backoff(fn, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            return fn()
        except RateLimitError:
            if attempt == max_attempts - 1:
                raise
            wait = (2 ** attempt) + random.random()
            time.sleep(wait)
```

In [ ]:
# Simulación de rate limiting y backoff exponencial
import time
import random

class RateLimitSimulator:
    """Simula un servidor con rate limit."""
    def __init__(self, max_requests=5, window_seconds=10):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.requests = []
    
    def make_request(self):
        """Intenta hacer una petición. Lanza error si excede el límite."""
        now = time.time()
        # Limpiar requests antiguos fuera de la ventana
        self.requests = [t for t in self.requests if now - t < self.window_seconds]
        
        if len(self.requests) >= self.max_requests:
            raise Exception(f"429 Too Many Requests (límite: {self.max_requests}/{self.window_seconds}s)")
        
        self.requests.append(now)
        return f"✅ Request exitoso ({len(self.requests)}/{self.max_requests})"

# Crear simulador con límite de 5 requests por 10 segundos
server = RateLimitSimulator(max_requests=5, window_seconds=10)

print("Simulación de rate limiting:\\n")
print(f"Límite: {server.max_requests} requests por {server.window_seconds} segundos\\n")

# Intentar hacer 8 requests rápidamente
for i in range(8):
    try:
        result = server.make_request()
        print(f"Request {i+1}: {result}")
        time.sleep(0.5)  # pequeña pausa entre requests
    except Exception as e:
        print(f"Request {i+1}: ❌ {e}")
        print(f"           → Esperando 2 segundos antes de reintentar...")
        time.sleep(2)
        # Reintentar
        try:
            result = server.make_request()
            print(f"           → Reintento exitoso: {result}")
        except Exception as e2:
            print(f"           → Reintento falló: {e2}")

print("\\n" + "="*70)
print("Visualización de backoff exponencial")
print("="*70)

# Mostrar cómo crece el tiempo de espera
fig, ax = plt.subplots(figsize=(10, 4))
attempts = range(1, 8)
wait_times = [2**i for i in range(len(attempts))]

ax.bar(attempts, wait_times, color='steelblue', alpha=0.8, edgecolor='white')
for i, (att, wait) in enumerate(zip(attempts, wait_times)):
    ax.text(att, wait + 2, f'{wait}s', ha='center', fontweight='bold')

ax.set_xlabel('Número de intento')
ax.set_ylabel('Tiempo de espera (segundos)')
ax.set_title('Backoff exponencial: tiempo de espera crece exponencialmente\\n(2^intento segundos)')
ax.set_xticks(attempts)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\\n💡 Lecciones clave:")
print("   • Rate limits protegen la infraestructura del proveedor")
print("   • Backoff exponencial evita saturar el servidor con reintentos")
print("   • Agregar jitter aleatorio previene 'thundering herd' (muchos clientes")
print("     reintentando al mismo tiempo)")


---
<a id='17'></a>
## 17. Métricas de similitud vectorial — comparando embeddings

Cuando tenemos dos vectores (embeddings), necesitamos medir qué tan "parecidos" son. Hay varias métricas, cada una con propiedades distintas.

### 1. Similitud coseno (la más usada)

Mide el **ángulo** entre dos vectores, ignorando su magnitud:

$$
\text{sim}_{\cos}(a, b) = \frac{a \cdot b}{\|a\| \|b\|} = \cos(\theta)
$$

**Propiedades:**
- Rango: [-1, 1] (en embeddings de texto, típicamente [0, 1])
- **Invariante a escala:** `sim(a, b) = sim(2a, b)` — solo importa la dirección
- **Cuándo usarla:** Casi siempre — es el estándar para embeddings

**Interpretación:**
- 1.0 → vectores idénticos (mismo ángulo)
- 0.9 → muy similares (sinónimos)
- 0.5 → algo relacionados
- 0.0 → ortogonales (sin relación)
- -1.0 → opuestos (antónimos)

### 2. Distancia euclidiana

Mide la **distancia en línea recta** entre dos puntos:

$$
d_{\text{eucl}}(a, b) = \sqrt{\sum_{i=1}^d (a_i - b_i)^2} = \|a - b\|
$$

**Propiedades:**
- Rango: [0, ∞)
- **Sensible a magnitud:** `d(a, b) ≠ d(2a, 2b)`
- **Cuándo usarla:** Cuando la magnitud del vector tiene significado (raro en embeddings)

**Conversión a similitud:**

$$
\text{sim}_{\text{eucl}}(a, b) = \frac{1}{1 + d_{\text{eucl}}(a, b)}
$$

### 3. Producto punto (dot product)

Combina similitud angular y magnitud:

$$
\text{sim}_{\text{dot}}(a, b) = a \cdot b = \sum_{i=1}^d a_i b_i
$$

**Propiedades:**
- Rango: [-∞, ∞]
- **Más rápido** que coseno (no requiere normalización)
- **Cuándo usarlo:** Cuando los embeddings ya están normalizados (magnitud = 1)

**Relación con coseno:**

Si $\|a\| = \|b\| = 1$ (vectores normalizados), entonces:

$$
\text{sim}_{\text{dot}}(a, b) = \text{sim}_{\cos}(a, b)
$$

### 4. Distancia de Manhattan (L1)

Suma de diferencias absolutas:

$$
d_{\text{L1}}(a, b) = \sum_{i=1}^d |a_i - b_i|
$$

**Cuándo usarla:** Rara vez en embeddings — más común en features categóricas.

### Comparación práctica

| Métrica | Velocidad | Invariante a escala | Uso en embeddings |
|---|---|---|---|
| **Coseno** | Media | ✅ | ⭐⭐⭐⭐⭐ (default) |
| **Euclidiana** | Media | ❌ | ⭐⭐ (ocasional) |
| **Dot product** | Rápida | ❌ | ⭐⭐⭐⭐ (si normalizados) |
| **Manhattan** | Media | ❌ | ⭐ (raro) |

### Normalización de vectores

Para convertir un vector a magnitud 1:

$$
\hat{a} = \frac{a}{\|a\|}
$$

Después de normalizar, `dot(a, b) = cosine(a, b)`, lo cual acelera cálculos.

In [ ]:
# Comparación de métricas de similitud
from scipy.spatial.distance import cosine as cosine_dist, euclidean

# Tres vectores de ejemplo (3D para visualizar)
a = np.array([1.0, 2.0, 1.0])
b = np.array([1.5, 2.5, 1.2])  # similar a 'a'
c = np.array([0.5, 0.2, 0.1])  # diferente de 'a'

print("Vectores de ejemplo:")
print(f"a = {a}")
print(f"b = {b}  (similar a 'a')")
print(f"c = {c}  (diferente de 'a')\\n")

# Calcular todas las métricas
def calcular_metricas(v1, v2, nombre1='a', nombre2='b'):
    # Similitud coseno (scipy devuelve distancia, así que restamos de 1)
    sim_cos = 1 - cosine_dist(v1, v2)
    
    # Distancia euclidiana
    dist_eucl = euclidean(v1, v2)
    sim_eucl = 1 / (1 + dist_eucl)  # convertir a similitud
    
    # Producto punto
    dot_prod = np.dot(v1, v2)
    
    # Manhattan
    dist_l1 = np.sum(np.abs(v1 - v2))
    
    print(f"Comparando {nombre1} vs {nombre2}:")
    print(f"  Similitud coseno:     {sim_cos:.4f}  (rango [0,1], más alto = más similar)")
    print(f"  Distancia euclidiana: {dist_eucl:.4f}  (rango [0,∞), más bajo = más similar)")
    print(f"  Similitud euclidiana: {sim_eucl:.4f}  (convertida a [0,1])")
    print(f"  Producto punto:       {dot_prod:.4f}  (rango [-∞,∞))")
    print(f"  Distancia Manhattan:  {dist_l1:.4f}  (rango [0,∞))\\n")
    
    return sim_cos, dist_eucl, dot_prod

print("="*70)
sim_ab_cos, dist_ab_eucl, dot_ab = calcular_metricas(a, b, 'a', 'b')
print("="*70)
sim_ac_cos, dist_ac_eucl, dot_ac = calcular_metricas(a, c, 'a', 'c')
print("="*70)

# Visualización 3D
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 5))

# Subplot 1: Vectores en 3D
ax1 = fig.add_subplot(121, projection='3d')
origin = [0, 0, 0]

# Dibujar vectores
ax1.quiver(*origin, *a, color='blue', arrow_length_ratio=0.1, linewidth=2, label='a')
ax1.quiver(*origin, *b, color='green', arrow_length_ratio=0.1, linewidth=2, label='b (similar)')
ax1.quiver(*origin, *c, color='red', arrow_length_ratio=0.1, linewidth=2, label='c (diferente)')

# Líneas de distancia euclidiana
ax1.plot([a[0], b[0]], [a[1], b[1]], [a[2], b[2]], 'g--', alpha=0.5, label=f'd(a,b)={dist_ab_eucl:.2f}')
ax1.plot([a[0], c[0]], [a[1], c[1]], [a[2], c[2]], 'r--', alpha=0.5, label=f'd(a,c)={dist_ac_eucl:.2f}')

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Vectores en 3D\\nLíneas punteadas = distancia euclidiana')
ax1.legend()

# Subplot 2: Comparación de métricas
ax2 = fig.add_subplot(122)

metricas = ['Coseno\\n(sim)', 'Euclidiana\\n(1/(1+d))', 'Dot\\nProduct']
valores_ab = [sim_ab_cos, 1/(1+dist_ab_eucl), dot_ab/10]  # dot/10 para escalar
valores_ac = [sim_ac_cos, 1/(1+dist_ac_eucl), dot_ac/10]

x = np.arange(len(metricas))
width = 0.35

bars1 = ax2.bar(x - width/2, valores_ab, width, label='a vs b (similar)', color='green', alpha=0.8)
bars2 = ax2.bar(x + width/2, valores_ac, width, label='a vs c (diferente)', color='red', alpha=0.8)

ax2.set_ylabel('Valor (normalizado)')
ax2.set_title('Comparación de métricas\\n(valores más altos = más similares)')
ax2.set_xticks(x)
ax2.set_xticklabels(metricas)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Anotar valores
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print("\\n💡 Observaciones:")
print("   • Coseno captura bien la similitud angular (b está cerca de a)")
print("   • Euclidiana es sensible a la magnitud (c está muy lejos de a)")
print("   • Dot product combina ángulo y magnitud")
print("   • Para embeddings, COSENO es casi siempre la mejor opción")


---
<a id='18'></a>
## 18. Vector Databases — búsqueda eficiente de embeddings

**Definición:** Una **vector database** es una base de datos optimizada para almacenar y buscar vectores de alta dimensionalidad (embeddings). Permite encontrar los **k vectores más cercanos** (k-NN) a un query en milisegundos, incluso con millones de vectores.

### El problema: búsqueda naive es lenta

Con búsqueda exhaustiva (comparar el query contra todos los vectores):

$$
\text{Complejidad} = O(n \cdot d)
$$

donde $n$ = número de vectores, $d$ = dimensionalidad.

**Ejemplo:** 1 millón de vectores de 1536 dims → 1.5 mil millones de operaciones por query 🐌

### Solución: índices aproximados (ANN)

**ANN (Approximate Nearest Neighbors)** sacrifica un poco de precisión por velocidad:

$$
\text{Complejidad} = O(\log n \cdot d) \text{ o mejor}
$$

**Algoritmos principales:**

| Algoritmo | Cómo funciona | Pros | Contras |
|---|---|---|---|
| **HNSW** | Grafo navegable jerárquico | Muy rápido, alta precisión | Usa mucha RAM |
| **IVF** | Divide el espacio en clusters | Escalable | Requiere entrenamiento |
| **LSH** | Hash sensible a localidad | Simple | Menos preciso |
| **FAISS** | Librería de Meta con múltiples índices | Muy optimizado | Complejo de configurar |

### Vector DBs populares

| Base de datos | Tipo | Cuándo usarla |
|---|---|---|
| **Pinecone** | Managed cloud | Producción, sin infra |
| **Weaviate** | Open-source | Self-hosted, flexible |
| **Qdrant** | Open-source | Rust, muy rápido |
| **Milvus** | Open-source | Escala masiva |
| **ChromaDB** | Open-source | Prototipado local |
| **pgvector** | Extensión PostgreSQL | Ya usas Postgres |

### Operaciones básicas

**1. Insertar vectores:**

```python
# Pseudocódigo
db.insert(
    id='doc_123',
    vector=[0.1, 0.2, ..., 0.5],  # 1536 dims
    metadata={'title': 'Documento X', 'author': 'Y'}
)
```

**2. Buscar similares:**

```python
results = db.search(
    query_vector=[0.15, 0.22, ..., 0.48],
    top_k=5,
    metric='cosine'
)
# Devuelve los 5 vectores más cercanos con sus metadatos
```

**3. Filtrado híbrido:**

```python
# Buscar vectores similares + filtro por metadata
results = db.search(
    query_vector=...,
    top_k=5,
    filter={'author': 'Y', 'year': {'$gte': 2020}}
)
```

### Métricas de performance

**Recall@k:** ¿Qué % de los verdaderos top-k encontramos?

$$
\text{Recall@k} = \frac{\text{# verdaderos positivos en top-k}}{k}
$$

**Latencia:** Tiempo de respuesta (típicamente <10 ms para millones de vectores).

**Throughput:** Queries por segundo (QPS).

### Cuándo NO necesitas una vector DB

- **<10k vectores:** Usa numpy + sklearn (búsqueda exhaustiva es suficiente)
- **Batch processing:** Calcula similitudes offline, guarda en SQL
- **Prototipo rápido:** ChromaDB en memoria

### Cuándo SÍ necesitas una vector DB

- **>100k vectores:** Búsqueda exhaustiva es muy lenta
- **Latencia crítica:** Necesitas respuestas en <100 ms
- **Actualizaciones frecuentes:** Insertar/borrar vectores dinámicamente
- **Producción:** RAG, búsqueda semántica, recomendaciones

In [ ]:
# Comparación: búsqueda naive vs índice optimizado
import time
from sklearn.metrics.pairwise import cosine_similarity

# Simular un corpus de embeddings
np.random.seed(42)
n_docs = 10000
dim = 1536  # dimensión típica de text-embedding-3-small

print(f"Generando {n_docs:,} vectores de {dim} dimensiones...")
corpus_vectors = np.random.randn(n_docs, dim).astype('float32')

# Normalizar para que dot product = cosine similarity
corpus_vectors = corpus_vectors / np.linalg.norm(corpus_vectors, axis=1, keepdims=True)

# Query vector
query_vector = np.random.randn(1, dim).astype('float32')
query_vector = query_vector / np.linalg.norm(query_vector)

print(f"Corpus: {corpus_vectors.shape}")
print(f"Query:  {query_vector.shape}\\n")

# ──────────────────────────────────────────────────────────────────────────────
# Método 1: Búsqueda NAIVE (exhaustiva) con cosine_similarity
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("MÉTODO 1: Búsqueda NAIVE (exhaustiva)")
print("="*70)

start = time.time()
similarities = cosine_similarity(query_vector, corpus_vectors)[0]
top_k_indices = np.argsort(similarities)[::-1][:5]
top_k_scores = similarities[top_k_indices]
time_naive = time.time() - start

print(f"Tiempo: {time_naive*1000:.2f} ms")
print(f"Top-5 índices: {top_k_indices}")
print(f"Top-5 scores:  {top_k_scores}\\n")

# ──────────────────────────────────────────────────────────────────────────────
# Método 2: Índice optimizado con FAISS (si está disponible)
# ──────────────────────────────────────────────────────────────────────────────
try:
    import faiss
    
    print("="*70)
    print("MÉTODO 2: Índice FAISS (optimizado)")
    print("="*70)
    
    # Crear índice FAISS (Flat = exhaustivo pero optimizado con SIMD)
    index = faiss.IndexFlatIP(dim)  # IP = Inner Product (dot product)
    index.add(corpus_vectors)
    
    start = time.time()
    distances, indices = index.search(query_vector, k=5)
    time_faiss = time.time() - start
    
    print(f"Tiempo: {time_faiss*1000:.2f} ms")
    print(f"Top-5 índices: {indices[0]}")
    print(f"Top-5 scores:  {distances[0]}")
    print(f"\\n⚡ Speedup: {time_naive/time_faiss:.1f}x más rápido\\n")
    
    # Verificar que los resultados son idénticos
    assert np.allclose(indices[0], top_k_indices), "Resultados no coinciden"
    print("✅ Verificación: ambos métodos devuelven los mismos resultados\\n")
    
except ImportError:
    print("="*70)
    print("MÉTODO 2: FAISS no disponible")
    print("="*70)
    print("Para instalar: pip install faiss-cpu\\n")
    time_faiss = None

# ──────────────────────────────────────────────────────────────────────────────
# Visualización: escalabilidad
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ANÁLISIS DE ESCALABILIDAD")
print("="*70)

# Simular diferentes tamaños de corpus
corpus_sizes = [1000, 5000, 10000, 50000, 100000]
times_naive = []
times_faiss = []

for size in corpus_sizes:
    # Generar corpus
    vecs = np.random.randn(size, 128).astype('float32')  # dim más chica para velocidad
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    q = np.random.randn(1, 128).astype('float32')
    q = q / np.linalg.norm(q)
    
    # Naive
    start = time.time()
    sims = cosine_similarity(q, vecs)[0]
    _ = np.argsort(sims)[::-1][:5]
    times_naive.append((time.time() - start) * 1000)
    
    # FAISS (si está disponible)
    try:
        import faiss
        idx = faiss.IndexFlatIP(128)
        idx.add(vecs)
        start = time.time()
        _, _ = idx.search(q, k=5)
        times_faiss.append((time.time() - start) * 1000)
    except:
        times_faiss.append(None)

# Graficar
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(corpus_sizes, times_naive, 'o-', label='Naive (exhaustiva)', linewidth=2, markersize=8)
if all(t is not None for t in times_faiss):
    ax.plot(corpus_sizes, times_faiss, 's-', label='FAISS (optimizada)', linewidth=2, markersize=8)

ax.set_xlabel('Tamaño del corpus (# vectores)')
ax.set_ylabel('Tiempo de búsqueda (ms)')
ax.set_title('Escalabilidad: Naive vs FAISS\\n(dim=128, top-k=5)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
ax.set_yscale('log')

# Anotar valores
for i, size in enumerate(corpus_sizes):
    ax.text(size, times_naive[i]*1.2, f'{times_naive[i]:.1f}ms', 
            ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print("\\n💡 Conclusiones:")
print("   • Búsqueda naive: O(n·d) — crece linealmente con el corpus")
print("   • FAISS optimizado: ~10-100x más rápido (SIMD, multithreading)")
print("   • Para >100k vectores, necesitas ANN (HNSW, IVF) para latencia <10ms")
print("   • Vector DBs (Pinecone, Weaviate) usan estos algoritmos internamente")


---
<a id='19'></a>
## 19. Caching — evitar trabajo redundante

**Definición:** Un **cache** es un almacenamiento temporal de resultados de operaciones costosas. Si la misma entrada se repite, devolvemos el resultado guardado en lugar de recalcular.

### ¿Por qué cachear?

| Operación | Costo sin cache | Costo con cache | Ahorro |
|---|---|---|---|
| Llamada a API LLM | $0.00005 + 100ms | 0 + 1ms | 100x velocidad, 100% costo |
| Generar embedding | $0.000001 + 50ms | 0 + 1ms | 50x velocidad, 100% costo |
| Búsqueda en vector DB | 10ms | 1ms | 10x velocidad |

### Tipos de cache

**1. Cache en memoria (in-memory)**

```python
cache = {}  # dict simple

def get_embedding_cached(text):
    if text in cache:
        return cache[text]
    
    embedding = generate_embedding(text)  # operación costosa
    cache[text] = embedding
    return embedding
```

**Pros:** Muy rápido (nanosegundos)  
**Contras:** Se pierde al reiniciar, limitado por RAM

**2. Cache en disco**

```python
import diskcache

cache = diskcache.Cache('./cache_dir')

@cache.memoize()
def get_embedding_cached(text):
    return generate_embedding(text)
```

**Pros:** Persiste entre reinicios  
**Contras:** Más lento que memoria (~1ms)

**3. Cache distribuido (Redis, Memcached)**

```python
import redis

r = redis.Redis(host='localhost', port=6379)

def get_embedding_cached(text):
    key = f'emb:{hash(text)}'
    cached = r.get(key)
    
    if cached:
        return pickle.loads(cached)
    
    embedding = generate_embedding(text)
    r.setex(key, 3600, pickle.dumps(embedding))  # TTL 1 hora
    return embedding
```

**Pros:** Compartido entre múltiples servidores, escalable  
**Contras:** Latencia de red (~1-5ms)

### Estrategias de invalidación

**1. TTL (Time To Live)**

Cada entrada expira después de un tiempo:

```python
cache.set('key', value, ttl=3600)  # expira en 1 hora
```

**2. LRU (Least Recently Used)**

Cuando el cache se llena, borra las entradas menos usadas:

```python
from functools import lru_cache

@lru_cache(maxsize=1000)  # guarda máximo 1000 entradas
def expensive_function(x):
    return x ** 2
```

**3. Invalidación manual**

Borras explícitamente cuando los datos cambian:

```python
cache.delete('user:123:profile')
```

### Cache de prompts (OpenAI, Anthropic)

Los proveedores cachean automáticamente partes del prompt que se repiten:

**OpenAI:** Cachea los primeros ~1024 tokens si se repiten en <5 min (50% descuento)

**Anthropic:** Marca explícitamente qué cachear:

```python
messages = [{
    'role': 'user',
    'content': [
        {'type': 'text', 'text': '<CONTEXTO LARGO>',
         'cache_control': {'type': 'ephemeral'}},  # ← cachea esto
        {'type': 'text', 'text': 'Pregunta específica'}
    ]
}]
```

El contexto cacheado cuesta ~10% del precio normal.

### Métricas de cache

**Hit rate:** % de requests que encuentran el resultado en cache

$$
\text{Hit rate} = \frac{\text{cache hits}}{\text{total requests}} \times 100\%
$$

**Objetivo:** >80% para que valga la pena

**Miss penalty:** Costo adicional de un cache miss (buscar en cache + calcular)

### Cuándo NO cachear

- **Datos que cambian frecuentemente:** Precios en tiempo real, noticias
- **Inputs únicos:** Cada query es diferente (ej. chat sin repetición)
- **Operaciones baratas:** Si calcular cuesta <1ms, el cache agrega overhead

### Cuándo SÍ cachear

- **Operaciones costosas:** APIs, embeddings, búsquedas complejas
- **Inputs repetitivos:** FAQs, búsquedas populares
- **Producción:** Reducir latencia y costo

In [ ]:
# Demostración de caching con diferentes estrategias
import hashlib
import time
from functools import lru_cache

# Simulamos una operación costosa (ej. llamada a API)
def operacion_costosa(input_text):
    """Simula una operación que toma 100ms."""
    time.sleep(0.1)  # 100ms de latencia
    return f"Resultado para: {input_text}"

# ──────────────────────────────────────────────────────────────────────────────
# Estrategia 1: Sin cache (baseline)
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ESTRATEGIA 1: Sin cache")
print("="*70)

queries = ["¿Qué es Python?", "¿Qué es ML?", "¿Qué es Python?", "¿Qué es ML?"]

start = time.time()
for q in queries:
    resultado = operacion_costosa(q)
    print(f"Query: '{q}' → {resultado[:30]}...")
tiempo_sin_cache = time.time() - start

print(f"\nTiempo total: {tiempo_sin_cache:.2f}s")
print(f"Promedio por query: {tiempo_sin_cache/len(queries)*1000:.0f}ms\n")

# ──────────────────────────────────────────────────────────────────────────────
# Estrategia 2: Cache en memoria (dict)
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ESTRATEGIA 2: Cache en memoria (dict)")
print("="*70)

cache_memoria = {}
cache_hits = 0
cache_misses = 0

def operacion_con_cache(input_text):
    global cache_hits, cache_misses
    
    # Usar hash como key (más eficiente que string largo)
    cache_key = hashlib.md5(input_text.encode()).hexdigest()
    
    if cache_key in cache_memoria:
        cache_hits += 1
        print(f"  ✅ CACHE HIT")
        return cache_memoria[cache_key]
    
    cache_misses += 1
    print(f"  ❌ CACHE MISS — calculando...")
    resultado = operacion_costosa(input_text)
    cache_memoria[cache_key] = resultado
    return resultado

start = time.time()
for q in queries:
    print(f"Query: '{q}'")
    resultado = operacion_con_cache(q)
tiempo_con_cache = time.time() - start

hit_rate = cache_hits / len(queries) * 100
print(f"\nTiempo total: {tiempo_con_cache:.2f}s")
print(f"Promedio por query: {tiempo_con_cache/len(queries)*1000:.0f}ms")
print(f"Cache hits: {cache_hits}, misses: {cache_misses}")
print(f"Hit rate: {hit_rate:.0f}%")
print(f"⚡ Speedup: {tiempo_sin_cache/tiempo_con_cache:.1f}x más rápido\n")

# ──────────────────────────────────────────────────────────────────────────────
# Estrategia 3: LRU Cache (con límite de tamaño)
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ESTRATEGIA 3: LRU Cache (límite de 2 entradas)")
print("="*70)

@lru_cache(maxsize=2)
def operacion_lru(input_text):
    print(f"  ❌ CACHE MISS — calculando '{input_text}'...")
    time.sleep(0.1)
    return f"Resultado para: {input_text}"

# Queries que exceden el límite del cache
queries_lru = ["A", "B", "C", "A", "B", "C"]

start = time.time()
for q in queries_lru:
    print(f"Query: '{q}'")
    resultado = operacion_lru(q)
tiempo_lru = time.time() - start

print(f"\nTiempo total: {tiempo_lru:.2f}s")
print(f"Cache info: {operacion_lru.cache_info()}")
print(f"Hit rate: {operacion_lru.cache_info().hits / len(queries_lru) * 100:.0f}%")
print("\n💡 Nota: Con maxsize=2, cuando llega 'C', se borra 'A' (LRU).")
print("   Por eso el segundo 'A' es cache miss.\n")

# ──────────────────────────────────────────────────────────────────────────────
# Visualización: comparación de estrategias
# ──────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Tiempo por estrategia
estrategias = ['Sin cache', 'Con cache\\n(dict)', 'LRU cache\\n(maxsize=2)']
tiempos = [tiempo_sin_cache, tiempo_con_cache, tiempo_lru]
colores = ['tomato', 'green', 'steelblue']

bars = axes[0].bar(estrategias, tiempos, color=colores, alpha=0.8, edgecolor='white')
for bar, t in zip(bars, tiempos):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{t:.2f}s', ha='center', fontweight='bold')

axes[0].set_ylabel('Tiempo total (segundos)')
axes[0].set_title('Comparación de estrategias de caching\\n(4 queries, 2 únicas)')
axes[0].grid(True, alpha=0.3, axis='y')

# Subplot 2: Hit rate
hit_rates = [0, hit_rate, operacion_lru.cache_info().hits / len(queries_lru) * 100]
bars = axes[1].bar(estrategias, hit_rates, color=colores, alpha=0.8, edgecolor='white')
for bar, hr in zip(bars, hit_rates):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{hr:.0f}%', ha='center', fontweight='bold')

axes[1].set_ylabel('Hit rate (%)')
axes[1].set_title('Tasa de aciertos del cache')
axes[1].set_ylim(0, 110)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────────────────────────────────────
# Análisis de ahorro de costo
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ANÁLISIS DE AHORRO DE COSTO")
print("="*70)

# Supongamos que cada query cuesta $0.0001 (ejemplo)
costo_por_query = 0.0001
queries_por_dia = 10000

# Sin cache
costo_sin_cache = queries_por_dia * costo_por_query

# Con cache (50% hit rate típico)
hit_rate_real = 0.5
queries_reales = queries_por_dia * (1 - hit_rate_real)
costo_con_cache = queries_reales * costo_por_query

ahorro_diario = costo_sin_cache - costo_con_cache
ahorro_mensual = ahorro_diario * 30

print(f"Escenario: {queries_por_dia:,} queries/día, ${costo_por_query:.4f} por query")
print(f"\nSin cache:")
print(f"  Costo diario:  ${costo_sin_cache:.2f}")
print(f"  Costo mensual: ${costo_sin_cache * 30:.2f}")
print(f"\nCon cache (hit rate {hit_rate_real*100:.0f}%):")
print(f"  Costo diario:  ${costo_con_cache:.2f}")
print(f"  Costo mensual: ${costo_con_cache * 30:.2f}")
print(f"\n💰 AHORRO:")
print(f"  Diario:  ${ahorro_diario:.2f} ({ahorro_diario/costo_sin_cache*100:.0f}%)")
print(f"  Mensual: ${ahorro_mensual:.2f}")
print(f"  Anual:   ${ahorro_mensual * 12:.2f}")

print("\n💡 Conclusiones:")
print("   • Cache en memoria: hit rate alto, pero se pierde al reiniciar")
print("   • LRU cache: limita uso de memoria, borra entradas viejas")
print("   • Para producción: Redis/Memcached (persistente, distribuido)")
print("   • Objetivo: hit rate >80% para que valga la pena")
print("   • Ahorro típico: 50-90% en costo y latencia")


---
<a id='1'></a>
## 1. Tokens y Tokenización

**Definición:** La **tokenización** es el proceso de dividir texto en unidades pequeñas llamadas **tokens**. Un token puede ser una palabra completa, una subpalabra, un carácter o incluso un byte.

### ¿Por qué no usar palabras completas?

| Problema | Solución con subpalabras |
|---|---|
| Vocabulario gigante (millones de palabras) | Vocabulario manejable (30k-200k tokens) |
| Palabras fuera del vocabulario (OOV) | Cualquier palabra se puede representar |
| No maneja morfología (correr, corriendo, corrió) | Comparte raíz: `[corr][iendo]`, `[corr][ió]` |

### Algoritmos principales

**1. Byte-Pair Encoding (BPE)** — usado por GPT, LLaMA
- Empieza con caracteres individuales
- Iterativamente fusiona los pares más frecuentes
- Ejemplo: `"running"` → `["run", "ning"]` o `["runn", "ing"]` según frecuencia

**2. WordPiece** — usado por BERT
- Similar a BPE pero maximiza la verosimilitud del corpus
- Usa `##` para marcar continuaciones: `["run", "##ning"]`

**3. SentencePiece** — usado por T5, LLaMA
- Trata el texto como secuencia de bytes (no requiere pre-tokenización)
- Soporta cualquier idioma sin espacios (chino, japonés)

### Propiedades importantes

- **Vocabulario fijo:** típicamente 30k-200k tokens
- **Reversible:** tokens → texto original (casi siempre)
- **Dependiente del modelo:** cada modelo tiene su propio tokenizer
- **Impacto en costo:** más tokens = más caro (APIs cobran por token)